# 018: Kmerseek vs OrthoFinder — MGI JAX Ground Truth

Compare kmerseek (HP k=24) and OrthoFinder on human-mouse ortholog detection using **MGI JAX** known orthologs as ground truth.

Evaluations:
- **Kmerseek** scored by Poisson p-value and jaccard
- **OrthoFinder** binary predictions
- Metrics: Precision, Recall, F1, AUC-ROC, AUC-PR
- Resource usage: wall time, peak memory

In [1]:
import gzip
import re
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import poisson
from sklearn.metrics import (
    RocCurveDisplay,
    average_precision_score,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from statsmodels.stats.multitest import multipletests

## Configuration

In [2]:
DATA_DIR = Path("/Users/olga/data/gencode/results-human-mouse-orthologs")
OF_DIR = Path("/Users/olga/data/gencode/data-for-orthofinder/OrthoFinder/Results_Mar03")

KSIZE = 24
SAMPLE_FRAC = 0.05
RANDOM_STATE = 42
ALPHA = 0.05

KMERSEEK_TSV = DATA_DIR / f"ortholog_evaluation.hp.k{KSIZE}.tsv.gz"
KMERSEEK_SEARCH_LOG = DATA_DIR / f"human_vs_mouse.hp.k{KSIZE}.search.log.gz"
HUMAN_INDEX_LOG = DATA_DIR / f"gencode.v49.pc_translations.fa.hp.k{KSIZE}.scaled1.kmerseek.index.log.gz"
# Human index log only goes to k22; use the .canonical mouse index for k24
MOUSE_INDEX_LOG = DATA_DIR / f"gencode.vM38.pc_translations.canonical.fa.hp.k{KSIZE}.scaled1.kmerseek.index.log.gz"

MGI_FILE = DATA_DIR / "HOM_MouseHumanSequence.rpt.gz"

OF_ORTHOLOGS_TSV = (
    OF_DIR
    / "Orthologues/Orthologues_gencode.v49.pc_translations"
    / "gencode.v49.pc_translations__v__gencode.vM38.pc_translations.tsv"
)

## 1. Load MGI JAX Ground Truth

`HOM_MouseHumanSequence.rpt` pairs rows by `DB Class Key`: one row per species per ortholog group.

In [3]:
mgi_raw = pd.read_csv(MGI_FILE, sep="\t", low_memory=False)
print(mgi_raw.shape)
mgi_raw.head(3)

(46522, 13)


,DB Class Key,Common Organism Name,NCBI Taxon ID,Symbol,EntrezGene ID,Mouse MGI ID,HGNC ID,OMIM Gene ID,Genetic Location,Genome Coordinates (mouse: GRCm39 human: GRCh38),Nucleotide RefSeq IDs,Protein RefSeq IDs,SWISS_PROT IDs
0,50916033,"mouse, laboratory",10090,Aldh1l1,107747,MGI:1340024,NaN,NaN,Chr6 40.16 cM,Chr6:90527751-90576153(+),"XM_030255052,NM_027406,NM_001356412","NP_001343341,NP_081682,XP_030110912",Q8R0Y6
1,50916033,human,9606,ALDH1L1,10840,NaN,HGNC:3978,OMIM:600249,Chr3 q21.3,Chr3:126103562-126197994(-),"NM_012190,NM_144776,NM_001270364,NM_001270365","XP_016861102,XP_011510657,NP_001257293,NP_0012...",O75891
2,50916034,"mouse, laboratory",10090,Sry,21674,MGI:98660,NaN,NaN,ChrY syntenic,ChrY:2662471-2663658(-),NM_011564,NP_035694,Q05738


In [4]:
# Pivot to (human_symbol, mouse_symbol) pairs per DB Class Key
human_rows = mgi_raw[mgi_raw["NCBI Taxon ID"] == 9606][["DB Class Key", "Symbol"]].rename(
    columns={"Symbol": "human_symbol"}
)
mouse_rows = mgi_raw[mgi_raw["NCBI Taxon ID"] == 10090][["DB Class Key", "Symbol"]].rename(
    columns={"Symbol": "mouse_symbol"}
)

mgi_pairs = human_rows.merge(mouse_rows, on="DB Class Key")

# Normalise to uppercase for matching
mgi_pairs["human_upper"] = mgi_pairs["human_symbol"].str.upper()
mgi_pairs["mouse_upper"] = mgi_pairs["mouse_symbol"].str.upper()

mgi_ortholog_set = set(zip(mgi_pairs["human_upper"], mgi_pairs["mouse_upper"]))
print(f"MGI ortholog pairs: {len(mgi_ortholog_set):,}")
mgi_pairs.head(3)

MGI ortholog pairs: 24,584


,DB Class Key,human_symbol,mouse_symbol,human_upper,mouse_upper
0,50916033,ALDH1L1,Aldh1l1,ALDH1L1,ALDH1L1
1,50916034,SRY,Sry,SRY,SRY
2,50916035,SOX12,Sox12,SOX12,SOX12


## 2. Load Kmerseek Data

In [ ]:
USECOLS = [
    "query_name", "target_name",
    "jaccard", "containment", "n_intersecting_hashes", "expected_shared_kmers",
    "prob_overlap", "enrichment", "query_tfidf",
    "human_gene", "mouse_gene", "is_ortholog",
]

df = pd.read_csv(
    KMERSEEK_TSV,
    sep="\t",
    usecols=USECOLS,
    low_memory=False,
).sample(frac=SAMPLE_FRAC, random_state=RANDOM_STATE)

print(f"Loaded: {len(df):,} rows  |  Ensembl orthologs: {df['is_ortholog'].sum():,}")

In [ ]:
# Poisson p-value: P(X >= k | lambda)
k = df["n_intersecting_hashes"].values
lam = df["expected_shared_kmers"].values
df["poisson_p"] = poisson.sf(k - 1, lam)

# Multiple-testing corrections
_, bh, _, _ = multipletests(df["poisson_p"], method="fdr_bh")
df["poisson_p_bh"] = bh
df["poisson_p_bonf"] = np.clip(df["poisson_p"] * len(df), 0, 1)

# MGI ground truth
df["human_upper"] = df["human_gene"].str.upper()
df["mouse_upper"] = df["mouse_gene"].str.upper()
df["is_mgi_ortholog"] = [
    (h, m) in mgi_ortholog_set for h, m in zip(df["human_upper"], df["mouse_upper"])
]

print(f"MGI orthologs in sample: {df['is_mgi_ortholog'].sum():,}")
print(f"Ensembl orthologs in sample: {df['is_ortholog'].sum():,}")

## 3. Load OrthoFinder Predictions

In [ ]:
def parse_gene_from_id(protein_id: str) -> str:
    """Extract gene symbol from GENCODE pipe-delimited protein ID.
    Format: prot|transcript|gene|...|transcript_name|gene_name|length
    """
    parts = protein_id.split("|")
    return parts[-2] if len(parts) >= 2 else protein_id


of_raw = pd.read_csv(OF_ORTHOLOGS_TSV, sep="\t")
human_col = "gencode.v49.pc_translations"
mouse_col = "gencode.vM38.pc_translations"

of_pairs = set()
for _, row in of_raw.iterrows():
    humans = [x.strip() for x in str(row[human_col]).split(",") if x.strip() and x.strip() != "nan"]
    mice = [x.strip() for x in str(row[mouse_col]).split(",") if x.strip() and x.strip() != "nan"]
    for h in humans:
        for m in mice:
            of_pairs.add((parse_gene_from_id(h).upper(), parse_gene_from_id(m).upper()))

print(f"OrthoFinder gene-level pairs: {len(of_pairs):,}")

In [ ]:
# Mark OrthoFinder predictions in the kmerseek sample
df["is_orthofinder"] = [
    (h, m) in of_pairs for h, m in zip(df["human_upper"], df["mouse_upper"])
]
print(f"OrthoFinder predictions in sample: {df['is_orthofinder'].sum():,}")

## 4. Evaluation Against MGI Ground Truth

Kmerseek thresholds:
- `pval`: raw Poisson p < 0.05
- `pval_bh`: BH-corrected p < 0.05
- `pval_bonf`: Bonferroni p < 0.05
- `jaccard`: jaccard > median jaccard of orthologs (data-driven)

OrthoFinder: binary predictions.

In [ ]:
def confusion_stats(y_true: pd.Series, y_pred: pd.Series) -> dict:
    tp = (y_true & y_pred).sum()
    fp = (~y_true & y_pred).sum()
    fn = (y_true & ~y_pred).sum()
    tn = (~y_true & ~y_pred).sum()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return dict(TP=int(tp), FP=int(fp), FN=int(fn), TN=int(tn),
                precision=precision, recall=recall, f1=f1,
                n_predicted=int(y_pred.sum()))


y_true = df["is_mgi_ortholog"]

# Jaccard threshold: 99th percentile of non-ortholog jaccard (data-driven stringent)
jaccard_threshold = df.loc[~y_true, "jaccard"].quantile(0.99)
print(f"Jaccard threshold (99th pct of non-orthologs): {jaccard_threshold:.4f}")

predictions = {
    "kmerseek_pval": df["poisson_p"] < ALPHA,
    "kmerseek_pval_bh": df["poisson_p_bh"] < ALPHA,
    "kmerseek_pval_bonf": df["poisson_p_bonf"] < ALPHA,
    f"kmerseek_jaccard>{jaccard_threshold:.4f}": df["jaccard"] > jaccard_threshold,
    "orthofinder": df["is_orthofinder"],
}

stats_rows = []
for name, y_pred in predictions.items():
    row = confusion_stats(y_true, y_pred)
    row["method"] = name
    stats_rows.append(row)

stats = pd.DataFrame(stats_rows).set_index("method")[
    ["n_predicted", "TP", "FP", "FN", "TN", "precision", "recall", "f1"]
]
stats.style.format({"precision": "{:.3f}", "recall": "{:.3f}", "f1": "{:.3f}"})

## 5. ROC and Precision-Recall Curves

In [ ]:
# Scoring functions for kmerseek continuous metrics
scores = {
    "kmerseek: -log10(p_raw)": -np.log10(df["poisson_p"].clip(lower=1e-300)),
    "kmerseek: -log10(p_BH)": -np.log10(df["poisson_p_bh"].clip(lower=1e-300)),
    "kmerseek: jaccard": df["jaccard"],
    "kmerseek: enrichment": df["enrichment"],
}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

colors = plt.cm.tab10.colors

# ROC
ax = axes[0]
for i, (label, score) in enumerate(scores.items()):
    fpr, tpr, _ = roc_curve(y_true, score)
    auc = roc_auc_score(y_true, score)
    ax.plot(fpr, tpr, color=colors[i], label=f"{label} (AUC={auc:.3f})")

# OrthoFinder as a point
of_stats = confusion_stats(y_true, df["is_orthofinder"])
of_fpr = of_stats["FP"] / (of_stats["FP"] + of_stats["TN"]) if (of_stats["FP"] + of_stats["TN"]) > 0 else 0
of_tpr = of_stats["recall"]
ax.scatter([of_fpr], [of_tpr], marker="*", s=200, color="black", zorder=5, label=f"OrthoFinder (TPR={of_tpr:.3f})")

ax.plot([0, 1], [0, 1], "k--", alpha=0.3)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC — MGI JAX Ground Truth")
ax.legend(fontsize=8, loc="lower right")

# Precision-Recall
ax = axes[1]
for i, (label, score) in enumerate(scores.items()):
    prec, rec, _ = precision_recall_curve(y_true, score)
    ap = average_precision_score(y_true, score)
    ax.plot(rec, prec, color=colors[i], label=f"{label} (AP={ap:.3f})")

# OrthoFinder as a point
ax.scatter([of_stats["recall"]], [of_stats["precision"]], marker="*", s=200, color="black", zorder=5,
           label=f"OrthoFinder (P={of_stats['precision']:.3f}, R={of_stats['recall']:.3f})")

baseline = y_true.mean()
ax.axhline(baseline, color="k", linestyle="--", alpha=0.3, label=f"Random baseline ({baseline:.4f})")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall — MGI JAX Ground Truth")
ax.legend(fontsize=8, loc="upper right")

plt.tight_layout()
plt.savefig(DATA_DIR / "vs_orthofinder_mgi_roc_pr.hp.k24.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. AUC Summary Table

In [ ]:
auc_rows = []
for label, score in scores.items():
    auc_rows.append({
        "Method": label,
        "AUC-ROC": roc_auc_score(y_true, score),
        "AUC-PR": average_precision_score(y_true, score),
    })

# OrthoFinder has no continuous score; use binary recall/precision
auc_rows.append({
    "Method": "OrthoFinder (binary)",
    "AUC-ROC": float("nan"),
    "AUC-PR": float("nan"),
})

auc_df = pd.DataFrame(auc_rows)
auc_df.style.format({"AUC-ROC": "{:.4f}", "AUC-PR": "{:.4f}"}).highlight_max(
    subset=["AUC-ROC", "AUC-PR"], color="lightgreen"
)

## 7. Threshold Sweep: F1 vs p-value

In [ ]:
thresholds = np.logspace(-20, 0, 200)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (col, label) in zip(axes, [
    ("poisson_p", "raw p-value"),
    ("poisson_p_bh", "BH-corrected p"),
    ("poisson_p_bonf", "Bonferroni p"),
]):
    f1s, precs, recs = [], [], []
    for t in thresholds:
        pred = df[col] < t
        s = confusion_stats(y_true, pred)
        f1s.append(s["f1"])
        precs.append(s["precision"])
        recs.append(s["recall"])

    log_t = np.log10(thresholds)
    ax.plot(log_t, f1s, label="F1", color="C0")
    ax.plot(log_t, precs, label="Precision", color="C1", linestyle="--")
    ax.plot(log_t, recs, label="Recall", color="C2", linestyle=":")
    ax.axvline(np.log10(ALPHA), color="grey", linestyle="-.", alpha=0.7, label=f"α={ALPHA}")

    best_idx = np.argmax(f1s)
    ax.scatter([log_t[best_idx]], [f1s[best_idx]], color="C0", zorder=5,
               label=f"Best F1={f1s[best_idx]:.3f} @ p<{thresholds[best_idx]:.2e}")

    ax.set_xlabel("log10(threshold)")
    ax.set_ylabel("Score")
    ax.set_title(label)
    ax.legend(fontsize=8)

plt.suptitle("Kmerseek threshold sweep — MGI JAX ground truth", y=1.01)
plt.tight_layout()
plt.savefig(DATA_DIR / "vs_orthofinder_mgi_threshold_sweep.hp.k24.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Ensembl vs MGI Ground Truth Comparison

How much do the two ground truths agree?

In [ ]:
gt_cross = pd.crosstab(
    df["is_ortholog"].map({True: "Ensembl ortholog", False: "Ensembl non-ortholog"}),
    df["is_mgi_ortholog"].map({True: "MGI ortholog", False: "MGI non-ortholog"}),
    margins=True,
)
print("Ground truth agreement:")
gt_cross

In [ ]:
# Recompute stats using Ensembl ground truth for comparison
y_ensembl = df["is_ortholog"]

rows = []
for gt_label, y_gt in [("MGI JAX", y_true), ("Ensembl", y_ensembl)]:
    for name, y_pred in predictions.items():
        s = confusion_stats(y_gt, y_pred)
        rows.append({"ground_truth": gt_label, "method": name,
                     "precision": s["precision"], "recall": s["recall"], "f1": s["f1"]})

compare_df = pd.DataFrame(rows).pivot(index="method", columns="ground_truth", values=["precision", "recall", "f1"])
compare_df.style.format("{:.3f}")

## 9. Resource Usage

In [ ]:
def parse_log_times(log_path: Path) -> dict:
    """Extract start time, end time, wall time, and peak memory from a kmerseek log."""
    with gzip.open(log_path, "rt") as f:
        content = f.read()

    start = re.search(r"Start time: (\S+ \S+)", content)
    end = re.search(r"End time: (\S+ \S+)", content)
    wall = re.search(r"([\d.]+) real", content)
    peak_mem = re.search(r"([\d]+)  peak memory footprint", content)
    max_rss = re.search(r"([\d]+)  maximum resident set size", content)

    result = {"log": log_path.name}
    if start and end:
        t0 = datetime.strptime(start.group(1), "%Y-%m-%d %H:%M:%S")
        t1 = datetime.strptime(end.group(1), "%Y-%m-%d %H:%M:%S")
        result["start"] = t0
        result["end"] = t1
        result["wall_s"] = (t1 - t0).total_seconds()
    if wall:
        result["wall_s"] = float(wall.group(1))  # override with precise value if available
    if peak_mem:
        result["peak_mem_gb"] = int(peak_mem.group(1)) / 1e9
    if max_rss:
        result["max_rss_gb"] = int(max_rss.group(1)) / 1e9
    return result


search_info = parse_log_times(KMERSEEK_SEARCH_LOG)
mouse_index_info = parse_log_times(MOUSE_INDEX_LOG)

# Human index only available up to k22; extrapolate or note N/A
human_index_wall_s = None
if HUMAN_INDEX_LOG.exists():
    human_index_info = parse_log_times(HUMAN_INDEX_LOG)
    human_index_wall_s = human_index_info.get("wall_s")

kmerseek_total_s = (
    search_info.get("wall_s", 0)
    + mouse_index_info.get("wall_s", 0)
    + (human_index_wall_s or 0)
)

orthofinder_total_s = 6107.058459  # from OrthoFinder output (2026-03-03)

resource_rows = [
    {
        "Tool": "Kmerseek (HP k=24)",
        "Step": "Index human proteome",
        "Wall time (s)": human_index_wall_s,
        "Peak mem (GB)": human_index_info.get("peak_mem_gb") if HUMAN_INDEX_LOG.exists() else None,
    },
    {
        "Tool": "Kmerseek (HP k=24)",
        "Step": "Index mouse proteome",
        "Wall time (s)": mouse_index_info.get("wall_s"),
        "Peak mem (GB)": mouse_index_info.get("peak_mem_gb"),
    },
    {
        "Tool": "Kmerseek (HP k=24)",
        "Step": "Search human vs mouse",
        "Wall time (s)": search_info.get("wall_s"),
        "Peak mem (GB)": search_info.get("max_rss_gb"),
    },
    {
        "Tool": "Kmerseek (HP k=24)",
        "Step": "TOTAL",
        "Wall time (s)": kmerseek_total_s if kmerseek_total_s > 0 else None,
        "Peak mem (GB)": None,
    },
    {
        "Tool": "OrthoFinder v3.1.0",
        "Step": "TOTAL (DIAMOND + MCL + trees)",
        "Wall time (s)": orthofinder_total_s,
        "Peak mem (GB)": None,
    },
]

resource_df = pd.DataFrame(resource_rows)
resource_df["Wall time (min)"] = resource_df["Wall time (s)"] / 60
resource_df["Wall time (hr)"] = resource_df["Wall time (min)"] / 60
resource_df[["Tool", "Step", "Wall time (s)", "Wall time (min)", "Wall time (hr)", "Peak mem (GB)"]].style.format(
    {"Wall time (s)": "{:.1f}", "Wall time (min)": "{:.1f}", "Wall time (hr)": "{:.2f}", "Peak mem (GB)": "{:.1f}"}
)

In [ ]:
# Summary bar chart
total_rows = resource_df[resource_df["Step"] == "TOTAL"].copy()

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.barh(
    total_rows["Tool"],
    total_rows["Wall time (s)"] / 3600,
    color=["C0", "C1"],
)
for bar, val in zip(bars, total_rows["Wall time (s)"]):
    ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height() / 2,
            f"{val/3600:.1f} hr", va="center", fontsize=11)
ax.set_xlabel("Wall time (hours)")
ax.set_title("Total compute time: Kmerseek vs OrthoFinder")
plt.tight_layout()
plt.savefig(DATA_DIR / "vs_orthofinder_mgi_resource_time.hp.k24.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Summary

Print a concise comparison table.

In [ ]:
print("=" * 70)
print("KMERSEEK vs ORTHOFINDER — MGI JAX Ground Truth (HP k=24, 5% sample)")
print("=" * 70)
print(f"Total pairs evaluated : {len(df):,}")
print(f"MGI ortholog pairs    : {int(y_true.sum()):,} ({y_true.mean()*100:.2f}%)")
print()
print(stats.to_string(float_format="{:.3f}".format))
print()
print("AUC (continuous scores):")
print(auc_df.to_string(index=False, float_format="{:.4f}".format))
print()
print("Wall time:")
for _, r in total_rows.iterrows():
    if pd.notna(r["Wall time (s)"]):
        print(f"  {r['Tool']}: {r['Wall time (s)']/3600:.2f} hr ({r['Wall time (s)']:.0f} s)")
print("=" * 70)